## LASSO ON CLUSTERING DATASETS

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.feature_selection import SelectFromModel

Fetching OpenML dataset 300 (Isolet)...
✅ Dataset loaded: 7797 samples, 617 features.

--- 🔵 1. Baseline (All 617 Features) ---
ARI Score: 0.4833

--- 🔴 2. Running LASSO Selection ---
(Using L1 Logistic Regression to find class-defining features...)


c:\Users\shrey\Desktop\RAVEN\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\shrey\Desktop\RAVEN\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


LASSO selected 594 features (Discarded 23)

--- 🟡 Running K-Means on 594 LASSO features ---
ARI Score:        0.5225
Silhouette Score: 0.0740

--- 📊 Final Comparison ---
Baseline ARI: 0.4833 (All Features)
LASSO ARI:    0.5225 (Supervised Selection)
✅ Insight: Removing noise helped K-Means see the clusters better.


In [ ]:
DATASET_ID = 300
print(f"Fetching OpenML dataset {DATASET_ID} (Isolet)...")

dataset = fetch_openml(data_id=DATASET_ID, as_frame=False, parser='auto')
X_raw = dataset.data
y_raw = dataset.target


In [ ]:

# Convert targets to integers for ARI calculation
le = LabelEncoder()
y_numeric = le.fit_transform(y_raw)

print(f"Dataset loaded: {X_raw.shape[0]} samples, {X_raw.shape[1]} features.")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)


In [ ]:

# Run K-Means directly on everything
kmeans_base = KMeans(n_clusters=26, random_state=42, n_init=10)
labels_base = kmeans_base.fit_predict(X_scaled)

ari_base = adjusted_rand_score(y_numeric, labels_base)
print(f"ARI Score: {ari_base:.4f}")


In [ ]:

# We use C=0.1 to force stronger sparsity (fewer features)
lasso_selector = LogisticRegression(
    penalty='l1', 
    solver='liblinear', 
    C=0.4, 
    random_state=42,
    max_iter=5000,
    multi_class='ovr' 
)

lasso_selector.fit(X_scaled, y_numeric)

# Extract features where coefficient is NOT zero
model = SelectFromModel(lasso_selector, prefit=True)
X_lasso = model.transform(X_scaled)
n_features = X_lasso.shape[1]


In [ ]:

print(f"LASSO selected {n_features} features (Discarded {X_raw.shape[1] - n_features})")

kmeans_lasso = KMeans(n_clusters=26, random_state=42, n_init=10)
labels_lasso = kmeans_lasso.fit_predict(X_lasso)

ari_lasso = adjusted_rand_score(y_numeric, labels_lasso)
sil_lasso = silhouette_score(X_lasso, labels_lasso)


In [ ]:

print(f"ARI Score:        {ari_lasso:.4f}")
print(f"Silhouette Score: {sil_lasso:.4f}")
print(f"Baseline ARI: {ari_base:.4f} (All Features)")
print(f"LASSO ARI:    {ari_lasso:.4f} (Supervised Selection)")


More OpenML Experiments

In [ ]:
DATASET_ID = 1485
print(f"Fetching OpenML dataset {DATASET_ID} (Madelon)...")

dataset = fetch_openml(data_id=DATASET_ID, as_frame=True, parser='auto')
X_raw = dataset.data
y_raw = dataset.target

# Convert targets to integers (-1, 1) -> (0, 1)
le = LabelEncoder()
y_numeric = le.fit_transform(y_raw)

print(f"Dataset loaded: {X_raw.shape[0]} samples, {X_raw.shape[1]} features.")


Fetching OpenML dataset 45093 (Madelon)...
✅ Dataset loaded: 203 samples, 12600 features.

--- 🔵 1. Baseline (All Features) ---
ARI Score: 0.0340

--- 🔴 2. Running LASSO Selection ---


c:\Users\shrey\Desktop\RAVEN\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


LASSO selected 142 features (Discarded 12458)

--- 🟡 Running K-Means on 142 LASSO features ---
ARI Score:        0.3508
Silhouette Score: 0.3661

--- 📊 Final Comparison (Madelon) ---
Baseline ARI: 0.0340 (All 500 Feats)
LASSO ARI:    0.3508 (Top 142 Feats)


In [ ]:


# Scaling is mandatory for both LASSO and K-Means
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

kmeans_base = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_base = kmeans_base.fit_predict(X_scaled)

ari_base = adjusted_rand_score(y_numeric, labels_base)
print(f"ARI Score: {ari_base:.4f}")


In [ ]:

# Logistic Regression with L1 penalty = LASSO for classification
# C=0.01 is a strong regularization to force it to find ONLY the 5 real features
lasso_selector = LogisticRegression(
    penalty='l1', 
    solver='liblinear', 
    C=0.4, 
    random_state=42,
    max_iter=5000
)

lasso_selector.fit(X_scaled, y_numeric)

# Transform data to keep only non-zero coefficients
model = SelectFromModel(lasso_selector, prefit=True)
X_lasso = model.transform(X_scaled)
n_features = X_lasso.shape[1]

print(f"LASSO selected {n_features} features (Discarded {X_raw.shape[1] - n_features})")



In [ ]:


kmeans_lasso = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_lasso = kmeans_lasso.fit_predict(X_lasso)

ari_lasso = adjusted_rand_score(y_numeric, labels_lasso)
sil_lasso = silhouette_score(X_lasso, labels_lasso)


In [ ]:

print(f"ARI Score:        {ari_lasso:.4f}")
print(f"Silhouette Score: {sil_lasso:.4f}")
print(f"Baseline ARI: {ari_base:.4f} (All 500 Feats)")
print(f"LASSO ARI:    {ari_lasso:.4f} (Top {n_features} Feats)")

## TEXT Clustering

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.feature_selection import SelectFromModel
from scipy.sparse import issparse 


DATASET_ID = 41084
print(f"Fetching OpenML dataset {DATASET_ID} (Madelon)...")

dataset = fetch_openml(data_id=DATASET_ID, as_frame=False, parser='auto')
X_raw = dataset.data
y_raw = dataset.target

if issparse(X_raw):
    print("Detected Sparse Matrix. Converting to Dense array...")
    X_raw = X_raw.toarray() 


Fetching OpenML dataset 41084 (Madelon)...
✅ Dataset loaded: 575 samples, 10304 features.
   Scaling data...

--- 🔵 1. Baseline (All Features) ---
ARI Score: 0.0295
Silhouette Score: 0.2278

--- 🔴 2. Running LASSO Selection ---


c:\Users\shrey\Desktop\RAVEN\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


LASSO selected 928 features (Discarded 9376)

--- 🟡 Running K-Means on 928 LASSO features ---
ARI Score:        0.0464
Silhouette Score: 0.1910

--- 📊 Final Comparison (Madelon) ---
Baseline ARI: 0.0295 (All Features)
LASSO ARI:    0.0464 (Top 928 Feats)


In [ ]:

# Convert targets to integers (-1, 1) -> (0, 1)
le = LabelEncoder()
y_numeric = le.fit_transform(y_raw)

print(f" Dataset loaded: {X_raw.shape[0]} samples, {X_raw.shape[1]} features.")

print("   Scaling data...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

kmeans_base = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_base = kmeans_base.fit_predict(X_scaled)

ari_base = adjusted_rand_score(y_numeric, labels_base)
sil_base = silhouette_score(X_scaled, labels_base)
print(f"ARI Score: {ari_base:.4f}")
print(f"Silhouette Score: {sil_base:.4f}")


In [ ]:

lasso_selector = LogisticRegression(
    penalty='l1', 
    solver='liblinear', 
    C=0.4, 
    random_state=42,
    max_iter=5000
)

lasso_selector.fit(X_scaled, y_numeric)

# Transform data
model = SelectFromModel(lasso_selector, prefit=True)
X_lasso = model.transform(X_scaled)
n_features = X_lasso.shape[1]

print(f"LASSO selected {n_features} features (Discarded {X_raw.shape[1] - n_features})")


In [ ]:


kmeans_lasso = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_lasso = kmeans_lasso.fit_predict(X_lasso)

ari_lasso = adjusted_rand_score(y_numeric, labels_lasso)
sil_lasso = silhouette_score(X_lasso, labels_lasso)


In [ ]:

print(f"ARI Score:        {ari_lasso:.4f}")
print(f"Silhouette Score: {sil_lasso:.4f}") 
print(f"Baseline ARI: {ari_base:.4f} (All Features)")
print(f"LASSO ARI:    {ari_lasso:.4f} (Top {n_features} Feats)")

# LASSO CLUSTERING ON CSV

In [ ]:
FILE_PATH = "datasets/df_destx.csv"
N_CLUSTERS = 2
INITIAL_GENES = 100000
LASSO_C = 0.01
RANDOM_STATE = 42



Loading datasets/df_destx.csv...
Raw shape: (167, 726)
No label column detected.
Matrix orientation appears correct (Samples x Genes)
Final Processing Shape: (167, 717)
Using Top 717 High-Variance Genes

--- Generating Pseudo-Labels ---


ValueError: Input X contains NaN.
KMeans does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:

print(f"Loading {FILE_PATH}...")
df = pd.read_csv(FILE_PATH) 
print(f"Raw shape: {df.shape}")


In [ ]:
# Extract Labels if they exist (for ARI calculation later)
label_col = None
for col in ["type", "class", "label", "phenotype"]:
    if col in df.columns:
        label_col = col
        break

if label_col:
    y_true = LabelEncoder().fit_transform(df[label_col].astype(str))
    df = df.drop(columns=[label_col])
    print(f"Detected and dropped label column: '{label_col}'")
else:
    y_true = None
    print("No label column detected.")


In [ ]:

# Drop non-numeric columns (Crucial fix)
X_raw = df.select_dtypes(include=[np.number])

# Transpose if needed (Ensure Rows=Samples, Cols=Genes)
# Gene data is often Genes x Samples, so we flip if Rows > Cols
if X_raw.shape[0] > X_raw.shape[1]:
    X_raw = X_raw.T
    print("Transposed matrix to (Samples x Genes)")
else:
    print("Matrix orientation appears correct (Samples x Genes)")

print(f"Final Processing Shape: {X_raw.shape}")


In [ ]:

X_log = np.log2(X_raw + 1)

# Variance Filtering (Reduce dimensionality for speed)
variances = X_log.var(axis=0)
# Use the minimum of requested genes or available genes
n_genes_to_keep = min(INITIAL_GENES, X_log.shape[1])
top_genes = variances.nlargest(n_genes_to_keep).index
X_filtered = X_log[top_genes]

print(f"Using Top {X_filtered.shape[1]} High-Variance Genes")

# Scale (Mandatory for K-Means and LASSO)
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X_filtered),
    columns=X_filtered.columns,
    index=X_filtered.index
)


In [ ]:
print("\n Generating Pseudo-Labels ")
kmeans_init = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
pseudo_labels = kmeans_init.fit_predict(X_scaled)

sil_init = silhouette_score(X_scaled, pseudo_labels)
print(f"Initial Silhouette: {sil_init:.4f}")



In [ ]:

print("\n Running LASSO Selection")
# We train LASSO to predict the Pseudo-Labels we just created
lasso = LogisticRegression(
    C=LASSO_C,
    penalty='l1',
    solver='liblinear',
    random_state=RANDOM_STATE
)

selector = SelectFromModel(lasso)
selector.fit(X_scaled, pseudo_labels)

selected_genes = X_scaled.columns[selector.get_support()]
X_lasso = X_scaled[selected_genes]

print(f"LASSO selected: {len(selected_genes)} genes")
print(f"Discarded:      {X_filtered.shape[1] - len(selected_genes)} genes")




In [ ]:

print("\n Final Clustering (LASSO Genes) ")

if len(selected_genes) > 0:
    kmeans_final = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
    labels_final = kmeans_final.fit_predict(X_lasso)

    sil_final = silhouette_score(X_lasso, labels_final)

    print(f"Final Silhouette: {sil_final:.4f}")
    
    # Calculate ARI if we found labels earlier
    if y_true is not None:
        ari = adjusted_rand_score(y_true, labels_final)
        print(f"ARI Score:  {ari:.4f}")

    # Summary
    print("\n SUMMARY ")
    print(f"Baseline (Top {X_filtered.shape[1]}): {sil_init:.4f}")
    print(f"LASSO    (Top {len(selected_genes)}):    {sil_final:.4f}")
else:
    print(" LASSO dropped all genes. Try increasing LASSO_C (e.g. to 1.0 or 5.0)")

Clustering on more CSVs / TSVs

In [ ]:
FILE_PATH = "human_liver.tsv"
N_CLUSTERS = 2
TAU = 0.95
RAVEN_SAMPLE = 903
VARIANCE_KEEP = 4500


Building redundancy graph...
Identifying essential and redundant features...
METRIC               | BASELINE        | RAVEN          
------------------------------------------------------------
Features Used        | 4500            | 4              
Data Reduction       | 0%              | 99.9%
Silhouette Score     | 0.3198          | 0.8229
Time Taken (s)       | 0.44            | 430.70


In [ ]:

df = pd.read_csv(FILE_PATH, sep="\t", index_col=0)
X_raw = df.T

selector = VarianceThreshold(threshold=0)
X_var = selector.fit_transform(X_raw)
feat_names = X_raw.columns[selector.get_support()]
X_df = pd.DataFrame(X_var, columns=feat_names, index=X_raw.index)

X_log = np.log2(X_df + 1)

scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X_log),
    columns=X_df.columns,
    index=X_df.index
)


In [ ]:

variances = X_scaled.var(axis=0)
top_genes = variances.sort_values(ascending=False).head(VARIANCE_KEEP).index
X_scaled = X_scaled[top_genes]

t0 = time.time()
kmeans_base = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
labels_base = kmeans_base.fit_predict(X_scaled)
sil_base = silhouette_score(X_scaled, labels_base)
time_base = time.time() - t0


In [ ]:

t0 = time.time()
selected_genes, dropped_genes = raven(
    X_scaled,
    mode="df",
    sample_size=RAVEN_SAMPLE,
    tau=TAU
)

X_raven = X_scaled[selected_genes]

kmeans_raven = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
labels_raven = kmeans_raven.fit_predict(X_raven)
sil_raven = silhouette_score(X_raven, labels_raven)
time_raven = time.time() - t0

n_base = X_scaled.shape[1]
n_raven = X_raven.shape[1]
reduction = 100 - (n_raven / n_base * 100)


In [ ]:

print(f"{'METRIC':<20} | {'BASELINE':<15} | {'RAVEN':<15}")

print(f"{'Features Used':<20} | {n_base:<15} | {n_raven:<15}")
print(f"{'Data Reduction':<20} | {'0%':<15} | {reduction:.1f}%")
print(f"{'Silhouette Score':<20} | {sil_base:.4f}          | {sil_raven:.4f}")
print(f"{'Time Taken (s)':<20} | {time_base:.2f}            | {time_raven:.2f}")

